# AntMaze preference pairs: flipping/rolling onto its back in preferred vs. non-preferred segments

This notebook analyzes the D4RL AntMaze **preference** datasets in `data/antmaze/` whose filenames contain `nt` (the `*_pref_nt.hdf5` files).

Each file stores preference *pairs*. For pair $i$ there are two trajectory segments:

* segment 1 — `states[i]`  (shape `(T, 29)`)
* segment 2 — `states_2[i]` (shape `(T, 29)`)

and a one-hot `labels[i]` of length 2 indicating which segment is **preferred**:
`labels[i] == [1, 0]` → segment 1 preferred, `labels[i] == [0, 1]` → segment 2 preferred.

### Defining a “flip / roll onto its back”
The 29-dim AntMaze observation is the ant's `qpos` (15) followed by `qvel` (14). Indices **3:7** are the torso orientation **quaternion** `[w, x, y, z]` (MuJoCo order).

The world-frame z-component of the torso's local up-axis (its *uprightness*) is

$$\text{up} = 1 - 2\,(x^2 + y^2),$$

which equals $\cos\theta$ where $\theta$ is the tilt of the torso from vertical: `up = +1` when perfectly upright, `up = -1` when fully upside-down. The ant is considered to be **on its back** when `up < UP_THRESH = 0.0`, i.e. the torso's up-axis points into the lower hemisphere ($\theta > 90^\circ$ — more upside-down than upright).

Empirically the uprightness is strongly **bimodal** (mass near $+1$ or near $-1$, very little in between), so the count is robust to the exact threshold.

A segment **contains a flip** if `up < UP_THRESH` at **any** valid timestep.

### What we count
For every preference pair we classify it into one of:
* **preferred only** — the ant flips ≥1× in the *preferred* segment but not at all in the non-preferred segment,
* **non-preferred only** — flips ≥1× in the *non-preferred* segment but not at all in the preferred segment,
* **both** — flips ≥1× in *both* segments.

(Pairs with no flip in either segment are reported as `neither` for completeness.)

In [1]:
import glob
import os

import h5py
import numpy as np
import pandas as pd

# --- configuration ---
DATA_DIR = "../data/antmaze"
QUAT_SLICE = slice(3, 7)   # torso orientation quaternion [w, x, y, z] in the 29-dim obs
UP_THRESH = 0.0            # 'on its back' when uprightness 1 - 2(x^2 + y^2) < UP_THRESH

nt_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_pref_nt.hdf5")))
print(f"Found {len(nt_files)} 'nt' preference files:")
for fp in nt_files:
    print("  ", os.path.basename(fp))

Found 4 'nt' preference files:
   antmaze-large-diverse-v2_pref_nt.hdf5
   antmaze-large-play-v2_pref_nt.hdf5
   antmaze-medium-diverse-v2_pref_nt.hdf5
   antmaze-medium-play-v2_pref_nt.hdf5


In [2]:
def segment_flips(states, mask=None, quat_slice=QUAT_SLICE, up_thresh=UP_THRESH):
    """Boolean array (n_pairs,): True if the torso is on its back at >=1 valid timestep."""
    quat = states[..., quat_slice]                     # (n_pairs, T, 4) = [w, x, y, z]
    x, y = quat[..., 1], quat[..., 2]
    uprightness = 1.0 - 2.0 * (x ** 2 + y ** 2)        # world-z of torso up-axis
    on_back = uprightness < up_thresh                  # (n_pairs, T)
    if mask is not None:
        on_back = on_back & mask.astype(bool)          # ignore padded timesteps
    return on_back.any(axis=1)


def analyze(path):
    with h5py.File(path, "r") as f:
        states = f["states"][:]          # (N, T, 29) -> segment 1
        states_2 = f["states_2"][:]      # (N, T, 29) -> segment 2
        labels = f["labels"][:]          # (N, 2) one-hot
        mask = f["attn_mask"][:] if "attn_mask" in f else None
        mask_2 = f["attn_mask_2"][:] if "attn_mask_2" in f else None

    flip_1 = segment_flips(states, mask)        # seg 1 contains a flip
    flip_2 = segment_flips(states_2, mask_2)    # seg 2 contains a flip

    seg1_is_preferred = labels[:, 0] == 1
    pref_flip = np.where(seg1_is_preferred, flip_1, flip_2)
    nonpref_flip = np.where(seg1_is_preferred, flip_2, flip_1)

    return {
        "dataset": os.path.basename(path).replace("_pref_nt.hdf5", ""),
        "preferred_only": int(np.sum(pref_flip & ~nonpref_flip)),
        "non_preferred_only": int(np.sum(~pref_flip & nonpref_flip)),
        "both": int(np.sum(pref_flip & nonpref_flip)),
        "neither": int(np.sum(~pref_flip & ~nonpref_flip)),
        "total_pairs": int(len(labels)),
    }


results = pd.DataFrame([analyze(fp) for fp in nt_files])
results

,dataset,preferred_only,non_preferred_only,both,neither,total_pairs
0,antmaze-large-diverse-v2,12,274,10,438,734
1,antmaze-large-play-v2,19,85,7,252,363
2,antmaze-medium-diverse-v2,3,317,3,389,712
3,antmaze-medium-play-v2,5,196,1,310,512


### Summary table

* **`preferred_only`** — flip in the preferred segment only.
* **`non_preferred_only`** — flip in the non-preferred segment only.
* **`both`** — flip in both segments.

(`neither` and `total_pairs` are included for context.)

In [3]:
cols = ["dataset", "preferred_only", "non_preferred_only", "both", "neither", "total_pairs"]
summary = results[cols].copy()
print(f"Flip criterion: torso uprightness 1 - 2(x^2 + y^2) < {UP_THRESH}\n")
print(summary.to_string(index=False))

Flip criterion: torso uprightness 1 - 2(x^2 + y^2) < 0.0

                  dataset  preferred_only  non_preferred_only  both  neither  total_pairs
 antmaze-large-diverse-v2              12                 274    10      438          734
    antmaze-large-play-v2              19                  85     7      252          363
antmaze-medium-diverse-v2               3                 317     3      389          712
   antmaze-medium-play-v2               5                 196     1      310          512
